# AI Programming Foundations Project

Author: Sukh Sandhu

This notebook builds a reusable data workflow using the scikit-learn Wine recognition dataset.

## Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('data/wine_dataset.csv')
FIG_DIR = Path('figures')
OUT_DIR = Path('outputs')
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)


## Ingestion

In [2]:
raw = pd.read_csv(DATA_PATH)
print(raw.shape)
display(raw.head())


(178, 15)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target,target_name
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0,class_0


## Cleaning

In [3]:
def standardize_column_names(dataframe):
    """Return a copy with lowercase, underscore-separated column names."""
    cleaned = dataframe.copy()
    cleaned.columns = [
        c.strip().lower().replace(' ', '_').replace('/', '_per_')
        for c in cleaned.columns
    ]
    return cleaned

def remove_duplicate_rows(dataframe):
    """Return a copy with duplicate rows removed while preserving all columns."""
    return dataframe.drop_duplicates().copy()

def validate_nonnegative_measurements(dataframe, feature_columns):
    """Check that selected chemistry measurements are nonnegative and return a validated copy."""
    validated = dataframe.copy()
    for column in feature_columns:
        if (validated[column] < 0).any():
            raise ValueError(f'Negative values found in {column}')
    return validated

cleaned = standardize_column_names(raw)
cleaned = remove_duplicate_rows(cleaned)
feature_cols = [c for c in cleaned.columns if c not in ['target', 'target_name']]
cleaned = validate_nonnegative_measurements(cleaned, feature_cols)
print(cleaned.shape)
display(cleaned.isna().sum().to_frame('missing_values').T)


(178, 15)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_per_od315_of_diluted_wines,proline,target,target_name
missing_values,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Exploratory Analysis

In [4]:
def summarize_by_class(dataframe, group_col='target_name'):
    """Create class-level counts and averages for the main wine chemistry fields."""
    return dataframe.groupby(group_col).agg(
        count=('target', 'size'),
        mean_alcohol=('alcohol', 'mean'),
        mean_flavanoids=('flavanoids', 'mean'),
        mean_color_intensity=('color_intensity', 'mean')
    ).round(3)

class_summary = summarize_by_class(cleaned)
display(class_summary)
correlations = cleaned[feature_cols].corr(numeric_only=True)
top_corr = correlations.abs().where(~np.eye(len(correlations), dtype=bool)).stack().sort_values(ascending=False).head(5)
display(top_corr.to_frame('absolute_correlation'))


,count,mean_alcohol,mean_flavanoids,mean_color_intensity
target_name,,,,
class_0,59,13.745,2.982,5.528
class_1,71,12.279,2.081,3.087
class_2,48,13.154,0.781,7.396


,,absolute_correlation
total_phenols,flavanoids,0.864564
flavanoids,total_phenols,0.864564
od280_per_od315_of_diluted_wines,flavanoids,0.787194
flavanoids,od280_per_od315_of_diluted_wines,0.787194
total_phenols,od280_per_od315_of_diluted_wines,0.699949


## Visualizations

In [5]:
fig, ax = plt.subplots(figsize=(7, 4))
cleaned.boxplot(column='alcohol', by='target_name', ax=ax)
ax.set_title('Figure 1. Alcohol distribution by wine cultivar')
ax.set_xlabel('Cultivar')
ax.set_ylabel('Alcohol')
plt.suptitle('')
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure_1_alcohol_by_class.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name, group in cleaned.groupby('target_name'):
    ax.scatter(group['total_phenols'], group['flavanoids'], label=name, alpha=0.75)
ax.set_title('Figure 2. Phenols and flavanoids by cultivar')
ax.set_xlabel('Total phenols')
ax.set_ylabel('Flavanoids')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure_2_phenols_flavanoids.png', dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(correlations, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Figure 3. Feature correlation heatmap')
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=90, fontsize=7)
ax.set_yticklabels(feature_cols, fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / 'figure_3_correlation_heatmap.png', dpi=160)
plt.show()


C:\Users\SukhPc\AppData\Local\Temp\ipykernel_45084\1896088483.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\SukhPc\AppData\Local\Temp\ipykernel_45084\1896088483.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\SukhPc\AppData\Local\Temp\ipykernel_45084\1896088483.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

The wine dataset loaded correctly and required only light cleaning because it is already a tidy benchmark table. The class-level summaries show that cultivar groups differ across alcohol, phenol, flavanoid, and color-intensity measurements. The visualizations support future modeling because they identify separable chemistry patterns as well as correlated predictors that may need attention in model design. A limitation is that the benchmark does not represent every production region or modern measurement process, so conclusions should not be generalized beyond the sampled wines without validation.

In [6]:
metrics = {
    'rows': int(cleaned.shape[0]),
    'columns': int(cleaned.shape[1]),
    'class_counts': cleaned['target_name'].value_counts().to_dict(),
    'top_correlations': {f'{a} vs {b}': float(v) for (a, b), v in top_corr.items()},
    'highest_mean_alcohol_class': class_summary['mean_alcohol'].idxmax()
}
with open(OUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
metrics


{'rows': 178,
 'columns': 15,
 'class_counts': {'class_1': 71, 'class_0': 59, 'class_2': 48},
 'top_correlations': {'total_phenols vs flavanoids': 0.8645635000951147,
  'flavanoids vs total_phenols': 0.8645635000951147,
  'od280_per_od315_of_diluted_wines vs flavanoids': 0.787193901866951,
  'flavanoids vs od280_per_od315_of_diluted_wines': 0.787193901866951,
  'total_phenols vs od280_per_od315_of_diluted_wines': 0.6999493647911861},
 'highest_mean_alcohol_class': 'class_0'}